<a href="https://colab.research.google.com/github/kerik3/CU_red_ai/blob/main/task1_sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import numpy
import torch

def seedset(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seedset(42)
print()

In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.8 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files='kid_adult.jsonl', split='train')

print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

1489
{'prompt': 'Как работает камера видеонаблюдения? Ответ: Она ловит свет через стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.', 'kid': 'Камера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и сохраняет как видео в памяти.', 'adult': 'Свет проходит через объектив на светочувствительную матрицу, которая преобразует фотоны в электрические сигналы. Затем процессор кодирует эти данные в цифровой формат и сохраняет итоговый видеопоток на накопитель.'}


In [25]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTConfig, SFTTrainer

dataset = load_dataset('json', data_files='kid_adult.jsonl', split='train')

def format_prompts_fn(examples):
    texts = []
    for p, k in zip(examples['prompt'], examples['kid']):
        texts.append(f"<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n{k}<|im_end|>")
    return {"text": texts}

dataset = dataset.map(format_prompts_fn, batched=True, remove_columns=dataset.column_names)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.torch_dtype = torch.float16
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, peft_config)

training_args = SFTConfig(
    output_dir="./sft_results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    optim="paged_adamw_32bit",
    seed=42,
    fp16=False,
    bf16=False,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args
)

trainer.train()

trainer.model.save_pretrained("sft_adapter")
tokenizer.save_pretrained("sft_adapter")

Map:   0%|          | 0/1489 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.309257
20,1.654118
30,1.517963
40,1.476374
50,1.478945
60,1.413602
70,1.381238
80,1.357102
90,1.371164


Step,Training Loss
10,2.309257
20,1.654118
30,1.517963
40,1.476374
50,1.478945
60,1.413602
70,1.381238
80,1.357102
90,1.371164
100,1.362791


('sft_adapter/tokenizer_config.json',
 'sft_adapter/chat_template.jinja',
 'sft_adapter/tokenizer.json')